# SAVIA — Demo del DAO
## Sistema de Almacenamiento de Índices de Vegetación para Investigación Agronómica
### Proyecto Integrador Bases de Datos II · UNdeC 2026

Este notebook demuestra el funcionamiento del módulo `SaviaDAO` —
la capa de acceso a datos del sistema SAVIA.

El sistema soporta **múltiples clientes** (bodegas, productores, cooperativas)
sobre una base de datos compartida. Todas las consultas operativas reciben
un `cliente_id` que garantiza el aislamiento lógico entre organizaciones.

> **Prerequisito:** ejecutar `seed.py` antes de correr este notebook.

In [ ]:
from dao import SaviaDAO
from db_models import Observacion
from datetime import date
import matplotlib.pyplot as plt

dao = SaviaDAO()
print("Conexión establecida con MongoDB")
print(f"Base de datos: vinedos_chilecito")

## Resolución del cliente

El modelo multi-tenant requiere un `cliente_id` en todas las consultas operativas.
Aquí obtenemos el ID del cliente "Bodega El Cóndor" para usar en las demos siguientes.

In [ ]:
# Buscar el cliente en la colección para obtener su _id
# (el seed lo inserta; aquí lo recuperamos para usarlo como filtro)
cliente_doc = dao._clientes.find_one({"nombre": "Bodega El Cóndor"})
cliente_id = str(cliente_doc["_id"])

print(f"Cliente seleccionado : {cliente_doc['nombre']}")
print(f"Plan                 : {cliente_doc['plan']}")
print(f"cliente_id           : {cliente_id}")

In [ ]:
# Consulta por zona — filtrada por cliente
parcelas_nonogasta = dao.get_parcelas_por_zona(cliente_id, "Nonogasta")
print(f"Parcelas en Nonogasta: {len(parcelas_nonogasta)}")
for p in parcelas_nonogasta:
    print(f"  - {p['nombre']} | {p['variedad']} | {p['superficie_ha']} ha")

# Consulta por cultivo — filtrada por cliente
parcelas_vid = dao.get_parcelas_por_cultivo(cliente_id, "vid")
print(f"\nTotal parcelas de vid (Bodega El Cóndor): {len(parcelas_vid)}")
for p in parcelas_vid:
    print(f"  - {p['nombre']} ({p['zona']})")

In [ ]:
# Recuperar una parcela por id y registrar una observación nueva
parcela = parcelas_vid[0]
parcela_id = str(parcela["_id"])

print(f"Parcela seleccionada: {parcela['nombre']}")
print(f"ID: {parcela_id}")

# Registrar una observación nueva.
# El índice único {parcela_id, fecha} en setup_db.py impide duplicados:
# si esta celda se ejecuta más de una vez, la segunda inserción falla
# con DuplicateKeyError. Se captura para que el notebook sea re-ejecutable.
from pymongo.errors import DuplicateKeyError

try:
    obs_id = dao.registrar_observacion(
        Observacion(
            parcela_id=parcela_id,
            fecha=date(2024, 3, 18),
            ndvi=0.22,
            evi=0.16,
            ndwi=0.06,
            nubosidad_pct=2.1,
            fuente="Sentinel-2",
        )
    )
    print(f"\nObservación registrada con id: {obs_id}")
except DuplicateKeyError:
    print("\nObservación ya existente para (parcela_id, 2024-03-18) — índice único activo, no se insertó duplicado.")

In [ ]:
# Consultar serie temporal completa
observaciones = dao.get_observaciones(parcela_id)
print(f"Total observaciones: {len(observaciones)}")
print("\nPrimeras 3:")
for o in observaciones[:3]:
    print(f"  {o['fecha']} | NDVI: {o['ndvi']} | EVI: {o['evi']} | Nubosidad: {o['nubosidad_pct']}%")

# Consultar por rango de fechas
obs_verano = dao.get_observaciones(
    parcela_id,
    desde=date(2023, 12, 1),
    hasta=date(2024, 2, 28)
)
print(f"\nObservaciones en verano (dic-feb): {len(obs_verano)}")

In [ ]:
fechas = [o["fecha"] for o in observaciones]
ndvi_vals = [o["ndvi"] for o in observaciones]
evi_vals = [o["evi"] for o in observaciones]

plt.figure(figsize=(12, 5))
plt.plot(fechas, ndvi_vals, marker="o", label="NDVI", color="#2d6a4f", linewidth=2)
plt.plot(fechas, evi_vals, marker="s", label="EVI", color="#74c69d", linewidth=2)
plt.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="Umbral 0.5")
plt.title("Serie temporal de índices espectrales — Finca El Peñón (Nonogasta)")
plt.xlabel("Fecha")
plt.ylabel("Valor del índice")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# NDVI promedio por zona para la temporada 2023/2024 — filtrado por cliente
resultado = dao.get_ndvi_promedio_por_zona(cliente_id, "Nonogasta", "2023/2024")
for r in resultado:
    print(f"Zona: {r['_id']}")
    print(f"  NDVI promedio: {r['ndvi_promedio']:.4f}")
    print(f"  Total observaciones: {r['total_observaciones']}")

In [ ]:
# Campañas registradas
campanas = dao.get_campanas(parcela_id)
print(f"Historial de cosechas — Finca El Peñón")
print("-" * 45)
for c in campanas:
    print(f"Temporada: {c['temporada']}")
    print(f"  Cosecha:      {c['fecha_cosecha']}")
    print(f"  Rendimiento:  {c['rendimiento_kg_ha']} kg/ha")
    print(f"  Notas:        {c['notas']}")
    print()

## Alertas

La colección `alertas` registra eventos detectados automáticamente cuando un índice
espectral cae por debajo del umbral configurado para la parcela.

In [ ]:
# Consultar alertas activas de la parcela seleccionada
alertas = dao.get_alertas_activas_por_parcela(parcela_id)
print(f"Alertas activas para '{parcela['nombre']}': {len(alertas)}")
for a in alertas:
    print(f"  [{a['tipo']}] {a['fecha']} | {a['indice']}: {a['valor_detectado']} (umbral: {a['umbral']}) | estado: {a['estado']}")

if not alertas:
    print("  (sin alertas activas)")

## Reportes

La colección `reportes` almacena los documentos generados para el cliente.
A diferencia de las colecciones operativas, los reportes cuelgan directamente
del cliente y pueden cubrir múltiples parcelas de la misma organización.

In [ ]:
# Consultar reportes del cliente
reportes = dao.get_reportes_por_cliente(cliente_id)
print(f"Reportes de '{cliente_doc['nombre']}': {len(reportes)}")
for r in reportes:
    print(f"  [{r['tipo']}] {r['nombre']}")
    print(f"    Período: {r['periodo']['desde']} → {r['periodo']['hasta']}")
    print(f"    Parcelas incluidas: {len(r['parcelas_incluidas'])}")
    print(f"    Estado: {r['estado']}")
    if r.get('resumen'):
        print(f"    Resumen: NDVI prom={r['resumen'].get('ndvi_promedio')} | "
              f"alertas={r['resumen'].get('parcelas_en_alerta')} | "
              f"obs={r['resumen'].get('observaciones_procesadas')}")
    print()

dao.cerrar()
print("\nConexión cerrada.")

## Consultas Geoespaciales Nativas

Las siguientes celdas demuestran el uso del índice **2dsphere** sobre el campo `geometria` de la colección `parcelas`. Este índice fue declarado en `setup_db.py` y habilita dos tipos de consulta espacial sobre los polígonos GeoJSON de los viñedos:

- `$nearSphere` — parcelas dentro de un radio desde un punto
- `$geoWithin` + `$geometry` — parcelas dentro de un rectángulo geográfico

Ninguna de estas consultas requiere cálculos en Python: toda la lógica espacial la ejecuta MongoDB internamente usando el índice.

> Todas las consultas siguen recibiendo `cliente_id` para respetar el aislamiento multi-tenant.

In [ ]:
from dao import SaviaDAO

dao = SaviaDAO()

# Recuperar cliente_id para las consultas geoespaciales
cliente_doc = dao._clientes.find_one({"nombre": "Bodega El Cóndor"})
cliente_id = str(cliente_doc["_id"])

### Consulta 1 — `get_parcelas_cerca_de` · `$nearSphere`

Devuelve todas las parcelas cuya geometría se encuentra dentro del radio indicado desde un punto geográfico. Los resultados vienen **ordenados por distancia** al punto de consulta, de más cercana a más lejana.

**Query MongoDB generada internamente:**
```json
{
  "cliente_id": "<cliente_id>",
  "geometria": {
    "$nearSphere": {
      "$geometry": { "type": "Point", "coordinates": [-67.49, -29.01] },
      "$maxDistance": 8000
    }
  }
}
```

In [ ]:
# Punto de referencia: centro aproximado del departamento Chilecito

LAT_CHILECITO = -29.01
LON_CHILECITO = -67.49
RADIO_KM = 8

parcelas_cercanas = dao.get_parcelas_cerca_de(
    cliente_id=cliente_id,
    lat=LAT_CHILECITO,
    lon=LON_CHILECITO,
    radio_km=RADIO_KM
)

print(f"Parcelas de '{cliente_doc['nombre']}' dentro de {RADIO_KM} km del centro de Chilecito: {len(parcelas_cercanas)}\n")
for p in parcelas_cercanas:
    print(f"  {p['nombre']:30s}  zona={p['zona']:20s}  altitud={p.get('altitud_msnm', '—')} msnm")

In [ ]:
# Reducir el radio para ver cómo cambia el resultado

parcelas_radio_chico = dao.get_parcelas_cerca_de(
    cliente_id=cliente_id,
    lat=LAT_CHILECITO,
    lon=LON_CHILECITO,
    radio_km=3
)

print(f"Con radio 3 km → {len(parcelas_radio_chico)} parcela(s):")
for p in parcelas_radio_chico:
    print(f"  {p['nombre']}")

### Consulta 2 — `get_parcelas_en_bbox` · `$geoWithin + $geometry`

Devuelve todas las parcelas que caen dentro de un rectángulo geográfico definido por sus esquinas suroeste y noreste. Internamente construye un **Polygon GeoJSON** y usa `$geoWithin`, que es la forma correcta de hacer esta consulta sobre un índice 2dsphere.

> **Nota técnica:** `$box` no funciona con índice 2dsphere — requiere un índice 2d plano. `$geoWithin + $geometry` es la alternativa correcta para datos GeoJSON.

**Query MongoDB generada internamente:**
```json
{
  "cliente_id": "<cliente_id>",
  "geometria": {
    "$geoWithin": {
      "$geometry": {
        "type": "Polygon",
        "coordinates": [[
          [-67.52, -29.03], [-67.46, -29.03],
          [-67.46, -28.98], [-67.52, -28.98],
          [-67.52, -29.03]
        ]]
      }
    }
  }
}
```

In [ ]:
# Bounding box amplio: cubre todas las zonas del seed

parcelas_en_area = dao.get_parcelas_en_bbox(
    cliente_id=cliente_id,
    sw_lat=-29.05, sw_lon=-67.53,
    ne_lat=-28.98, ne_lon=-67.45
)

print(f"Parcelas de '{cliente_doc['nombre']}' en el área: {len(parcelas_en_area)}\n")
for p in parcelas_en_area:
    print(f"  {p['nombre']:30s}  zona={p['zona']}")

In [ ]:
# Bbox reducido: solo el sector norte (Famatina)
# lat > -29.00 excluye Nonogasta

parcelas_norte = dao.get_parcelas_en_bbox(
    cliente_id=cliente_id,
    sw_lat=-29.00, sw_lon=-67.53,
    ne_lat=-28.98, ne_lon=-67.45
)

print(f"Parcelas en el sector norte (Famatina): {len(parcelas_norte)}")
for p in parcelas_norte:
    print(f"  {p['nombre']}  —  {p['zona']}")

In [ ]:
dao.cerrar()